# run_all_0416b_nb10 — three-stage adaptive seller × fixed buyers

Part of the 0416b negotiation-data sweep (deadline-free version).

* Seller strategies covered : **6 three-stage schedules, boundaries @ rounds 6 and 11 of 15**
* Buyer strategies covered  : **all 10 fixed buyers**
* Mode                      : **adaptive seller × fixed buyer**
* Target per-notebook runtime : **~18 h** on `deepseek-reasoner`
* No round-limit or deadline information is included in any prompt.

The six seller strategies in the experiment universe are:
`anchor_high`, `cooperative`, `boulware_seller`, `conceder_seller`,
`tit_for_tat_matcher`, `relational`.

The ten buyer strategies are: `random`, `budget`, `wild`, `concession`,
`anchor_drag`, `boulware_buyer`, `conceder_buyer`, `tit_for_tat_buyer`,
`persistent_f0.30`, `persistent_f0.55`.


In [1]:
import os
# ── Imports ───────────────────────────────────────────────────────────
import sys
from pathlib import Path

HERE = Path.cwd()
for p in (HERE, HERE.parent, HERE.parent.parent):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from openai import OpenAI

import exp_runner_0416 as R
from run_all_0416 import run_experiment_block

from exp_runner_0416 import (
    RandomBuyer, BudgetBuyer, WildBuyer,
    ConcessionBuyer, AnchorDragBuyer,
    BoulwareBuyer, ConcederBuyer, TitForTatBuyer,
    PersistentBuyer, AdaptiveBuyer,
    fixed_seller_schedule, twostage_seller_schedule, threestage_seller_schedule,
)
from run_all_0416 import (
    all_fixed_buyers_for_notebook, fixed_pairs_for_sellers,
)


# ── Config (edit as needed) ──────────────────────────────────────────
CLIENT = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],  # Set this key in your environment.
    base_url="https://api.deepseek.com",
)
MODEL       = "deepseek-reasoner"
N_ROUNDS    = 15
BASE_SEED   = 5000 + 10000        # distinct seed offset per notebook
OUTPUT_ROOT = Path("results_0416b")
NOTEBOOK_ID = "nb10"


## EXPERIMENTS list — notebook 10

In [2]:
# Three-stage adaptive seller × 10 fixed buyers.
# 6 schedules × 10 buyers × 4 runs = 240 runs (~18 h).
# Stage boundaries at rounds 6 and 11 (evenly split 15 rounds into thirds).

THREE_STAGE_SCHEDULES = [
    ("anchor_high",        "cooperative",        "conceder_seller"),
    ("boulware_seller",    "tit_for_tat_matcher","conceder_seller"),
    ("cooperative",        "anchor_high",        "boulware_seller"),
    ("conceder_seller",    "cooperative",        "anchor_high"),
    ("relational",         "cooperative",        "boulware_seller"),
    ("tit_for_tat_matcher","anchor_high",        "cooperative"),
]

R1, R2 = 6, 11      # stage boundaries (1-indexed)
N_RUNS_PER_PAIR = 4

BUYERS = all_fixed_buyers_for_notebook()

EXPERIMENTS = []
for s1, s2, s3 in THREE_STAGE_SCHEDULES:
    sched = threestage_seller_schedule(s1, s2, s3, r1=R1, r2=R2, n_rounds=N_ROUNDS)
    for b in BUYERS:
        tag = f"sel3_{s1}__{s2}__{s3}__r{R1}r{R2}__buy_{b.name}"
        EXPERIMENTS.append({
            "tag": tag, "buyer": b, "seller_schedule": sched,
            "n_runs": N_RUNS_PER_PAIR,
        })

print(f"Total conditions: {len(EXPERIMENTS)}")
print(f"Total runs      : {sum(e['n_runs'] for e in EXPERIMENTS)}")


Total conditions: 60
Total runs      : 240


## Run the block

In [3]:
# ── Execute ───────────────────────────────────────────────────────────
manifest = run_experiment_block(
    experiments=EXPERIMENTS,
    notebook_id=NOTEBOOK_ID,
    client=CLIENT,
    model=MODEL,
    n_rounds=N_ROUNDS,
    base_seed=BASE_SEED,
    output_root=OUTPUT_ROOT,
    verbose=True,
    resume=True,   # skip conditions whose JSON already exists
)
print("\n--- Per-condition timings (minutes) ---")
for t in manifest["timings"]:
    print(t)



[1/60]  sel3_anchor_high__cooperative__conceder_seller__r6r11__buy_random
  buyer   = random
  seller  = anchor_high@1..cooperative@6..conceder_seller@11  (adaptive=True)
  n_runs  = 4   n_rounds = 15

  --- Run 1/4  seed=15000  prod=Sushi pack (small) cost=$2.56 mkt=$9.49 int=$5.69 ---
  R 1 BUY $8.25 T6 [buy=random]  SEL $9.25 [sel=anchor_high]  "Thank you for your interest. Given the high-quality ingredie..."
  R 2 BUY $8.89 T3 [buy=random]  SEL $9.10 [sel=anchor_high]  "I appreciate you moving closer to a workable price, but $8.8..."
  R 3 BUY $7.00 T4 [buy=random]  SEL $9.05 [sel=anchor_high]  "I understand you're looking for a competitive price, but $7...."
  R 4 BUY $4.44 T2 [buy=random]  SEL $9.00 [sel=anchor_high]  "I understand you're aiming for a very low price, but $4.44 p..."
  R 5 BUY $4.90 T0 [buy=random]  SEL $9.00 [sel=anchor_high]  "I appreciate your continued interest, but $4.90 per unit is ..."
  R 6 BUY $3.70 T7 [buy=random]  SEL $9.00 [sel=cooperative]  "I unders

## Inspection

In [4]:
# ── Quick inspection of the first run of the first condition ────────
import json
first_tag = EXPERIMENTS[0]["tag"]
with open(OUTPUT_ROOT / NOTEBOOK_ID / f"{first_tag}.json") as f:
    exp = json.load(f)
R.inspect_run(exp["runs"][0], max_chars=200)


UnicodeDecodeError: 'gbk' codec can't decode byte 0x94 in position 3970: illegal multibyte sequence